###  Purpose of the Code

This script performs **document reranking** using a lightweight LLM-based NLP toolkit called **WordLlama**. It takes an initial retrieval result from a DirichletLM smoothed model and refines the ranking using semantic similarity between query and document texts. The goal is to investigate the effectiveness of Wordllama method.

---

### What the Code Does

1. **Load Queries:**
   - Reads a CSV file containing topic queries with `qid` and `query` columns.

2. **Load Initial Ranking:**
   - Loads a CSV file containing the initial ranking results (e.g., top 100 documents per query) with columns like `qid`, `docno`, and `text`.

3. **Load WordLlama Model:**
   - Loads the `l3_supercat` configuration of the WordLlama model, which uses cosine similarity in a 1024-dimensional embedding space.

4. **Rerank Documents:**
   - For each query:
     - Extracts the corresponding top documents from the initial ranking.
     - Uses WordLlama’s `rank()` function to rerank these documents based on their semantic similarity to the query.
     - Stores the new ranking positions and similarity scores.
   - Returns a new DataFrame with updated `qid`, `docno`, `rank`, `score`, and `query`.




In [1]:
#%pip install pandas
#%pip install wordLlama
#%pip install numpy
#%pip install sklearn

### Wordllama 3

In [43]:
import pandas as pd
from tqdm import tqdm
from wordllama import WordLlama

# 1. Load the Queries
query_file_path = "/mnt/ceph/storage/data-tmp/2024/rewu8105/all_topics/webis-touche2020_v2_queries.csv"
topics = pd.read_csv(query_file_path)

print("Queries loaded:\n", topics.tail())

# 2. Load Initial Ranking File
rank_file_path = "/mnt/ceph/storage/data-tmp/2024/rewu8105/initial_rank_result_top_100/webis_touche_dirichlet_100.csv"
rank_df = pd.read_csv(rank_file_path)
print("Initial ranking file:\n", rank_df.head())

# 3. Load Llama 3-Based WordLlama Model
wl = WordLlama.load(config="l3_supercat", dim=1024, binary=False)  # Explicitly use L3_supercat model with cosine similarity

# 4. Rerank the Documents Based on Initial Ranking
def rerank_documents(topics, rank_df):
    reranked_results = []
    
    # Extract doc_texts from rank_df
    doc_texts = rank_df.set_index('docno')['text'].to_dict()

    # Loop through each query
    for _, query_row in tqdm(topics.iterrows(), total=len(topics), desc="Reranking Documents"):
        query_id = query_row['qid']
        query_text = query_row['query']
        
        # Extract the document IDs (docno) from the initial ranking for this query
        initial_ranking_subset = rank_df[rank_df['qid'] == query_id]
        doc_ids = initial_ranking_subset['docno'].tolist()
        
        # Extract document texts for the docnos in the initial ranking
        documents_to_rerank = {docno: doc_texts[docno] for docno in doc_ids if docno in doc_texts}
        
        # Prepare lists for WordLlama
        doc_texts_list = list(documents_to_rerank.values())
        docnos = list(documents_to_rerank.keys())
        
        # Rerank the documents using WordLlama
        reranked_docs = wl.rank(query_text, doc_texts_list)  # Rank the subset of documents based on similarity
        
        # Store the reranked results for this query
        for rerank, (doc_text, score) in enumerate(reranked_docs, start=1):
            docno = docnos[doc_texts_list.index(doc_text)]  # Find the corresponding docno for the text
            reranked_results.append({
                'qid': query_id,
                'docno': docno,
                'rank': rerank,
                'score': score,
                'query': query_text
            })

    # Convert the reranked results to a DataFrame
    return pd.DataFrame(reranked_results)

# 5. Usage
reranked_df = rerank_documents(topics, rank_df)
print(reranked_df.head())


Queries loaded:
     qid                                          query
44   46             Should net neutrality be restored?
45   47                        Is homework beneficial?
46   48              Should the voting age be lowered?
47   49   Should body cameras be mandatory for police?
48   50  Should everyone get a universal basic income?
Initial ranking file:
    qid   docid                                    docno  rank      score  \
0    1  143806  51530f3f-2019-04-18T18:15:02Z-00004-000     0  31.770993   
1    1  164415  b0680508-2019-04-18T13:48:51Z-00002-000     1  31.583122   
2    1    4619  c065954f-2019-04-18T14:32:52Z-00003-000     2  31.521730   
3    1    4617  c065954f-2019-04-18T14:32:52Z-00001-000     3  31.476861   
4    1  163479  ff0947ec-2019-04-18T12:23:12Z-00000-000     4  31.385355   

                        query  \
0  Should teachers get tenure   
1  Should teachers get tenure   
2  Should teachers get tenure   
3  Should teachers get tenure   
4  Shoul

Reranking Documents: 100%|██████████████████████████████████████████████████████████████| 49/49 [00:13<00:00,  3.70it/s]

   qid                                    docno  rank     score  \
0    1  51530f3f-2019-04-18T18:15:02Z-00004-000     1  0.774642   
1    1  c065954f-2019-04-18T14:32:52Z-00003-000     2  0.769261   
2    1  ff0947ec-2019-04-18T12:23:12Z-00000-000     3  0.762361   
3    1  b0680508-2019-04-18T13:48:51Z-00002-000     4  0.762200   
4    1  c065954f-2019-04-18T14:32:52Z-00000-000     5  0.762011   

                         query  
0  Should teachers get tenure?  
1  Should teachers get tenure?  
2  Should teachers get tenure?  
3  Should teachers get tenure?  
4  Should teachers get tenure?  


In [44]:
# 5. Save the Final Results to a CSV file
output_path = "/mnt/ceph/storage/data-tmp/2024/rewu8105/new_wordllama/wordllama3_webis.csv"
reranked_df.to_csv(output_path, index=False)

print("Final ranked results saved to", output_path)

Final ranked results saved to /mnt/ceph/storage/data-tmp/2024/rewu8105/new_wordllama/wordllama3_webis.csv
